# Member 3 — Classical ML: Repeat-Purchase Prediction

This notebook builds a leakage-aware customer repeat-purchase model. Predictors come only from the customer's **first observed order**; the target is whether the customer makes more than one observed invoice in the available dataset window.

**Track deliverables:** feature engineering, baseline comparison, hyperparameter tuning, stratified cross-validation, feature importance, business KPI simulation, and saved model artifact.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.inspection import permutation_importance
# Resolve project root reliably whether the notebook is launched from notebooks/ or the repo root.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not ((ROOT / 'src').is_dir() and (ROOT / 'data').is_dir()):
    ROOT = ROOT.parent
if not ((ROOT / 'src').is_dir() and (ROOT / 'data').is_dir()):
    raise RuntimeError(f'Could not find project root from {Path.cwd()}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
from src.features.engineering import build_repeat_purchase_dataset
from src.models.train import build_models, tune_random_forest, evaluate_fitted, save_artifact, DEFAULT_FEATURES

# Prefer the stable sample/processed file already produced by the data pipeline.
paths = [ROOT/'data/sample.csv', ROOT/'spark/final_processed_data/part-00000-c97e019c-7edf-4aee-aef8-90585fde7cb0-c000.csv']
DATA_PATH = next((p for p in paths if p.exists()), None)
assert DATA_PATH is not None, 'No processed retail CSV found.'
raw = pd.read_csv(DATA_PATH)
raw.head()

In [ ]:
dataset = build_repeat_purchase_dataset(raw)
print('Customer rows:', len(dataset))
print('Repeat rate:', round(dataset.repeat_customer.mean(), 3))
print(dataset[['observed_order_count','repeat_customer']].value_counts().sort_index())
dataset.describe().T

## 1. Feature engineering and leakage check

Only first-order variables are used as predictors. `observed_order_count` is retained for target construction but is excluded from `X`. This prevents later purchases from entering the feature set.

In [ ]:
X = dataset[DEFAULT_FEATURES].copy()
y = dataset['repeat_customer'].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print('Train repeat rate:', y_train.mean(), 'Test repeat rate:', y_test.mean())

## 2. Baseline models + cross-validation

Because repeat customers are a minority class, average precision, recall, F1 and balanced accuracy are reported in addition to accuracy.

In [ ]:
models = build_models()
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
rows=[]
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv,
                            scoring=['average_precision','roc_auc','f1','recall','balanced_accuracy'])
    rows.append({
        'model': name,
        **{metric: scores[f'test_{metric}'].mean() for metric in ['average_precision','roc_auc','f1','recall','balanced_accuracy']}
    })
baseline_results = pd.DataFrame(rows).sort_values('average_precision', ascending=False)
baseline_results

## 3. Hyperparameter tuning

Random Forest is tuned with 3-fold stratified CV. `average_precision` is used as the selection metric because the positive class is relatively rare.

In [ ]:
rf_search = tune_random_forest(X_train, y_train)
print('Best params:', rf_search.best_params_)
print('Best CV average precision:', rf_search.best_score_)
rf_metrics = evaluate_fitted(rf_search.best_estimator_, X_test, y_test)
pd.DataFrame([rf_metrics], index=['tuned_random_forest'])

## 4. Explainability — permutation importance

Permutation importance measures how much test-set scoring degrades when each feature is shuffled. It is model-agnostic and easier to reproduce in the deployment environment than a mandatory SHAP dependency.

In [ ]:
perm = permutation_importance(rf_search.best_estimator_, X_test, y_test,
                                scoring='average_precision', n_repeats=20, random_state=42)
importance = pd.DataFrame({'feature': X_test.columns,
                           'importance_mean': perm.importances_mean,
                           'importance_std': perm.importances_std}).sort_values('importance_mean', ascending=False)
importance

In [ ]:
ax = importance.head(10).plot.barh(x='feature', y='importance_mean', legend=False)
ax.set_title('Top permutation importances — tuned Random Forest')
ax.invert_yaxis()
plt.tight_layout()

## 5. Business KPI simulation

Illustrative scenario: a retention campaign is sent to the highest-probability customers. The simulation estimates incremental revenue from a configurable conversion uplift and average repeat-order value. These are scenario assumptions, not causal estimates.

In [ ]:
test_out = X_test.copy()
test_out['actual_repeat'] = y_test.values
test_out['p_repeat'] = rf_search.best_estimator_.predict_proba(X_test)[:,1]

CAMPAIGN_RATE = 0.20
CONVERSION_UPLIFT = 0.05
AVG_REPEAT_ORDER_VALUE = float(dataset.loc[dataset.repeat_customer.eq(1), 'first_order_revenue'].mean())
N_TARGETED = max(1, int(len(test_out) * CAMPAIGN_RATE))
top = test_out.nlargest(N_TARGETED, 'p_repeat')
expected_incremental_revenue = N_TARGETED * CONVERSION_UPLIFT * AVG_REPEAT_ORDER_VALUE
baseline_repeat_rate = float(y_test.mean())
targeted_observed_repeat_rate = float(top.actual_repeat.mean())

kpi = pd.Series({
    'customers_in_test': len(test_out),
    'customers_targeted': N_TARGETED,
    'baseline_repeat_rate': baseline_repeat_rate,
    'targeted_observed_repeat_rate': targeted_observed_repeat_rate,
    'avg_repeat_order_value_assumption': AVG_REPEAT_ORDER_VALUE,
    'assumed_conversion_uplift': CONVERSION_UPLIFT,
    'scenario_incremental_revenue': expected_incremental_revenue,
})
kpi

## 6. Final fit and artifact

After model selection, the tuned estimator is refit on the training split (already performed by `GridSearchCV`) and saved under `models/`. Member 6 can load the `.joblib` artifact for deployment and use the JSON metadata for model/version information.

In [ ]:
model_path, meta_path = save_artifact(
    rf_search.best_estimator_, DEFAULT_FEATURES, rf_metrics,
    output_dir=ROOT/'models', name='classical_repeat_purchase_rf_v1'
)
print(model_path)
print(meta_path)

### Limitations
- The supplied repository contains a small processed sample (300 transaction rows), so reported metrics are illustrative rather than production-grade.
- The repeat label means repeat behavior **within the observed sample/window**, not a guaranteed future purchase.
- The KPI simulation uses explicit assumptions and should not be interpreted as a causal business impact estimate.
- With the full Online Retail II dataset available, rerun the notebook to obtain production-scale estimates and consider time-based validation.